### Import Packages

In [ ]:
import pytest
from numpy import pi, sqrt

### Import Classes

In [3]:
from environment import Environment
from extended_kalman_filter import ExtendedKalmanFilter
from kalman_filter import KalmanFilter
from robot import Robot
from sensors import SensorInterface, WheelEncoder, LandmarkPinger, GPS
from utils import Position, Pose, Landmark, Bounds

### Environment Tests

In [ ]:
class TestEnvironmentInitialization:
    """
    Tests for Environment initialization.
    """

    @pytest.fixture
    def basic_env(self):
        """
        Fixture providing a basic environment.
        """
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[Bounds(2, 3, 2, 3)],
            landmarks=[Landmark(Position(7, 7), 0), Landmark(Position(3, 8), 1)],
            timestep=0.1,
        )

    def test_initialization_basic(self, basic_env: Environment):
        """
        Test the environment properly instantiates all attributes.
        """
        assert basic_env.DIMENSIONS == Bounds(0, 10, 0, 10)
        assert basic_env.DT == 0.1
        assert basic_env.time == 0.0
        assert basic_env.robot_pose == Pose(Position(5, 5), 0)
        assert len(basic_env.OBSTACLES) == 1
        assert len(basic_env.LANDMARKS) == 2

    def test_initialization_rejects_agent_outside_bounds(self):
        """
        Test initialization fails if agent starts outside environment.
        """
        with pytest.raises(AssertionError):
            Environment(
                dimensions=Bounds(0, 10, 0, 10),
                robot_pose=Pose(Position(15, 5), 0),  # Outside bounds
                obstacles=[],
                landmarks=[],
                timestep=0.1,
            )

    def test_initialization_rejects_obstacle_outside_bounds(self):
        """
        Test initialization fails if obstacle is outside environment.
        """
        with pytest.raises(AssertionError):
            Environment(
                dimensions=Bounds(0, 10, 0, 10),
                robot_pose=Pose(Position(5, 5), 0),
                obstacles=[Bounds(-1, 2, 2, 3)],  # Extends outside
                landmarks=[],
                timestep=0.1,
            )

    def test_initialization_rejects_landmark_outside_bounds(self):
        """
        Test initialization fails if landmark is outside environment.
        """
        with pytest.raises(AssertionError):
            Environment(
                dimensions=Bounds(0, 10, 0, 10),
                robot_pose=Pose(Position(5, 5), 0),
                obstacles=[],
                landmarks=[Landmark(Position(15, 15), 0)],  # Outside
                timestep=0.1,
            )

    def test_initialization_rejects_duplicate_landmark_ids(self):
        """
        Test initialization fails if landmarks have duplicate IDs.
        """
        with pytest.raises(AssertionError):
            Environment(
                dimensions=Bounds(0, 10, 0, 10),
                robot_pose=Pose(Position(5, 5), 0),
                obstacles=[],
                landmarks=[
                    Landmark(Position(2, 2), 0),
                    Landmark(Position(8, 8), 0),  # Duplicate ID
                ],
                timestep=0.1,
            )
            
class TestRobotStep:
    """
    Tests for moving robotic agents in the environment.
    """

    @pytest.fixture
    def env(self):
        """
        Fixture providing environment for robot_step tests.
        """
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[Bounds(2, 3, 2, 3)],
            landmarks=[Landmark(Position(7, 7), 0)],
            timestep=0.1,
        )

    def test_robot_step_updates_position(self, env: Environment):
        """
        Test robot_step updates agent position.
        """
        initial_x = env.robot_pose.pos.x
        initial_y = env.robot_pose.pos.y

        env.robot_step(dx=1.0, dy=0.5, dtheta=0)

        assert env.robot_pose.pos.x == initial_x + 1.0
        assert env.robot_pose.pos.y == initial_y + 0.5

    def test_robot_step_updates_heading(self, env):
        """
        Test robot_step updates agent heading.
        """
        initial_theta = env.robot_pose.theta
        dtheta = 0.5

        env.robot_step(dx=0, dy=0, dtheta=dtheta)

        assert env.robot_pose.theta == pytest.approx(initial_theta + dtheta)

    def test_robot_step_increments_time(self, env):
        """
        Test robot_step increments time by DT.
        """
        initial_time = env.time

        env.robot_step(dx=0, dy=0, dtheta=0)

        assert env.time == initial_time + env.DT

    def test_robot_step_wraps_heading(self, env):
        """
        Test robot_step wraps heading to [-π, π].
        """
        env.robot_pose = Pose(Position(5, 5), pi - 0.1)

        # Rotate past π
        env.robot_step(dx=0, dy=0, dtheta=0.5)

        # Should wrap to negative
        assert env.robot_pose.theta < -pi + 0.5
        assert env.robot_pose.theta >= -pi

    def test_robot_step_wraps_negative_heading(self, env):
        """
        Test robot_step wraps large negative heading.
        """
        env.robot_pose = Pose(Position(5, 5), -pi + 0.1)

        # Rotate past -π
        env.robot_step(dx=0, dy=0, dtheta=-0.5)

        # Should wrap to positive
        assert env.robot_pose.theta > pi - 0.5
        assert env.robot_pose.theta <= pi

    def test_robot_step_multiple_steps(self, env):
        """
        Test multiple robot steps accumulate correctly.
        """
        initial_time = env.time
        initial_x = env.robot_pose.pos.x

        steps = 10
        dx_per_step = 0.1

        for _ in range(steps):
            env.robot_step(dx=dx_per_step, dy=0, dtheta=0)

        assert env.time == pytest.approx(initial_time + steps * env.DT)
        assert env.robot_pose.pos.x == pytest.approx(initial_x + steps * dx_per_step)

class TestValidateXYMotion:
    """
    Tests for validating the robot's movement.
    """

    @pytest.fixture
    def env(self):
        """
        Fixture providing environment with obstacle.
        """
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[Bounds(2, 3, 2, 3)],
            landmarks=[],
            timestep=0.1,
        )

    def test_validate_xy_motion_accepts_valid_motion(self, env: Environment):
        """
        Test validate_xy_motion allows valid motion.
        """
        env.robot_pose = Pose(Position(5, 5), 0)

        new_pos = env.validate_xy_motion(dx=1.0, dy=1.0)

        assert new_pos.x == 6.0
        assert new_pos.y == 6.0

    def test_validate_xy_motion_blocks_x_into_obstacle(self, env: Environment):
        """
        Test validate_xy_motion blocks x motion into obstacle.
        """
        # Position agent just before obstacle
        env.robot_pose = Pose(Position(1.5, 2.5), 0)
        # Obstacle is at Bounds(2, 3, 2, 3)

        # Try to move into obstacle
        new_pos = env.validate_xy_motion(dx=1.0, dy=0)

        # X should not have moved
        assert new_pos.x == 1.5
        assert new_pos.y == 2.5

    def test_validate_xy_motion_blocks_y_into_obstacle(self, env: Environment):
        """
        Test validate_xy_motion blocks y motion into obstacle.
        """
        # Position agent just before obstacle
        env.robot_pose = Pose(Position(2.5, 1.5), 0)
        # Obstacle is at Bounds(2, 3, 2, 3)

        # Try to move into obstacle
        new_pos = env.validate_xy_motion(dx=0, dy=1.0)

        # Y should not have moved
        assert new_pos.x == 2.5
        assert new_pos.y == 1.5

    def test_validate_xy_motion_allows_sliding_x(self, env: Environment):
        """
        Test validate_xy_motion allows sliding in x when y is blocked.
        """
        # Position agent at (2.5, 1.5) - y is at obstacle boundary
        env.robot_pose = Pose(Position(2.5, 1.5), 0)
        # Obstacle at Bounds(2, 3, 2, 3)

        # Try to move diagonally - x is clear, y would go into obstacle
        new_pos = env.validate_xy_motion(dx=0.3, dy=0.5)

        # X should move (valid from 2.5 to 2.8, not in obstacle)
        assert new_pos.x == pytest.approx(2.8)
        # Y should not move (1.5 + 0.5 = 2.0, which is inside obstacle y-range [2,3])
        assert new_pos.y == 1.5

    def test_validate_xy_motion_allows_sliding_y(self, env: Environment):
        """
        Test validate_xy_motion allows sliding in y when x is blocked.
        """
        # Position agent at (1.5, 2.5) - inside obstacle's y-range
        env.robot_pose = Pose(Position(1.5, 2.5), 0)
        # Obstacle at Bounds(2, 3, 2, 3)

        # Try to move diagonally - x would go into obstacle, y might be valid
        new_pos = env.validate_xy_motion(dx=0.6, dy=-0.6)

        # X should not move (1.5 + 0.6 = 2.1, which is inside obstacle x-range [2,3])
        assert new_pos.x == 1.5
        # Y should move (2.5 - 0.6 = 1.9, which is outside obstacle y-range [2,3])
        assert new_pos.y == pytest.approx(1.9)

    def test_validate_xy_motion_blocks_at_boundary(self, env: Environment):
        """
        Test validate_xy_motion prevents leaving environment bounds.
        """
        env.robot_pose = Pose(Position(9.5, 5), 0)
        # Environment is Bounds(0, 10, 0, 10)

        # Try to move past boundary
        new_pos = env.validate_xy_motion(dx=1.0, dy=0)

        # Should be blocked at boundary
        assert new_pos.x == 9.5

    def test_validate_xy_motion_negative_motion(self, env: Environment):
        """
        Test validate_xy_motion works with negative deltas.
        """
        env.robot_pose = Pose(Position(5, 5), 0)

        new_pos = env.validate_xy_motion(dx=-1.0, dy=-0.5)

        assert new_pos.x == 4.0
        assert new_pos.y == 4.5

    def test_validate_xy_motion_zero_motion(self, env: Environment):
        """
        Test validate_xy_motion handles zero motion.
        """
        env.robot_pose = Pose(Position(5, 5), 0)

        new_pos = env.validate_xy_motion(dx=0, dy=0)

        assert new_pos.x == 5.0
        assert new_pos.y == 5.0
        
class TestIsValidPos:
    """
    Tests for validating arbitrary positions in the environment.
    """

    @pytest.fixture
    def env(self):
        """Fixture providing environment with obstacle"""
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[Bounds(2, 3, 2, 3)],
            landmarks=[],
            timestep=0.1,
        )

    def test_is_valid_pos_accepts_free_space(self, env: Environment):
        """
        Test is_valid_pos returns True for valid position.
        """
        # Position (5, 5) is in free space
        assert env.is_valid_pos(Position(5, 5)) is True

    def test_is_valid_pos_rejects_obstacle(self, env: Environment):
        """
        Test is_valid_pos returns False for position in obstacle.
        """
        # Obstacle at Bounds(2, 3, 2, 3)
        assert env.is_valid_pos(Position(2.5, 2.5)) is False

    def test_is_valid_pos_rejects_outside_bounds(self, env: Environment):
        """
        Test is_valid_pos returns False for out-of-bounds positions.
        """
        for pos in [
            Position(-1, 5),
            Position(11, 5),
            Position(5, -1),
            Position(5, 11),
        ]:
            assert env.is_valid_pos(pos) is False

    def test_is_valid_pos_on_boundary(self, env: Environment):
        """
        Test is_valid_pos accepts positions exactly on boundary.
        """
        for pos in [
            Position(0, 5),
            Position(10, 5),
            Position(5, 0),
            Position(5, 10),
        ]:
            assert env.is_valid_pos(pos) is True

    def test_is_valid_pos_on_obstacle_edge(self, env: Environment):
        """
        Test is_valid_pos rejects positions on obstacle edge.
        """
        # Obstacle at Bounds(2, 3, 2, 3)
        # Edge is considered part of obstacle
        for pos in [
            Position(2, 2.5),
            Position(3, 2.5),
            Position(2.5, 2),
            Position(2.5, 3),
        ]:
            assert env.is_valid_pos(pos) is False


class TestGetGtRobotPose:
    """
    Tests for accessing ground truth robot pose.
    """

    @pytest.fixture
    def env(self):
        """
        Fixture providing basic environment.
        """
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[],
            timestep=0.1,
        )

    def test_get_gt_robot_pose_returns_current_pose(self, env: Environment):
        """
        Test get_gt_robot_pose returns current agent pose.
        """
        pose = env.get_gt_robot_pose()

        assert pose.pos.x == env.robot_pose.pos.x
        assert pose.pos.y == env.robot_pose.pos.y
        assert pose.theta == env.robot_pose.theta

    def test_get_gt_robot_pose_after_motion(self, env: Environment):
        """
        Test get_gt_robot_pose reflects motion.
        """
        env.robot_step(dx=1.0, dy=0.5, dtheta=0.3)

        pose = env.get_gt_robot_pose()

        assert pose.pos.x == 6.0
        assert pose.pos.y == 5.5
        assert pose.theta == pytest.approx(0.3)


class TestGetGtToLandmarks:
    """
    Tests for working with landmarks.
    """

    @pytest.fixture
    def env(self):
        """
        Fixture providing environment with landmarks.
        """
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(7, 7), 0), Landmark(Position(3, 8), 1)],
            timestep=0.1,
        )

    def test_get_landmark_by_id_found(self, env: Environment):
        """
        Test that an existing landmark ID returns the correct landmark.
        """
        lm = env.get_landmark_by_id(0)
        assert lm is not None
        assert lm.id == 0
        assert lm.pos == Position(7, 7)

    def test_get_landmark_by_id_not_found(self, env: Environment):
        """
        Test that a non-existent landmark ID returns None.
        """
        lm = env.get_landmark_by_id(99)
        assert lm is None

    def test_get_landmark_by_pos_single_match(self, env: Environment):
        """
        Test that querying a position with a single landmark returns a list of length one.
        """
        lms = env.get_landmarks_by_pos(Position(7, 7))
        assert len(lms) == 1
        assert lms[0].id == 0

    def test_get_landmark_by_pos_no_match(self, env: Environment):
        """
        Test that querying a position with no associated landmark returns an empty list.
        """
        lms = env.get_landmarks_by_pos(Position(0, 0))
        assert lms == []

    def test_get_landmark_by_pos_correct_landmark_returned(self, env: Environment):
        """
        Test that the returned landmark at a queried position has the expected ID.
        """
        lms = env.get_landmarks_by_pos(Position(3, 8))
        assert len(lms) == 1
        assert lms[0].id == 1

    def test_get_gt_to_landmarks_returns_dict(self, env: Environment):
        """
        Test get_gt_to_landmarks returns dictionary.
        """
        measurements = env.get_gt_to_landmarks()
        assert isinstance(measurements, dict)

    def test_get_gt_to_landmarks_has_all_landmarks(self, env: Environment):
        """
        Test get_gt_to_landmarks includes all landmarks.
        """
        measurements = env.get_gt_to_landmarks()

        assert len(measurements) == len(env.LANDMARKS)
        for landmark in env.LANDMARKS:
            assert landmark in measurements

    def test_get_gt_to_landmarks_correct_range(self, env: Environment):
        """
        Test get_gt_to_landmarks computes correct range.
        """
        # Agent at (5, 5), landmark at (7, 7)
        env.robot_pose = Pose(Position(5, 5), 0)

        measurements = env.get_gt_to_landmarks()

        # Find the single landmark at (7, 7)
        landmark_77 = env.get_landmarks_by_pos(Position(7, 7))[0]

        expected_range = sqrt((7 - 5) ** 2 + (7 - 5) ** 2)
        assert measurements[landmark_77].range == pytest.approx(expected_range)

    def test_get_gt_to_landmarks_correct_bearing_facing_north(self):
        """
        Test get_gt_to_landmarks computes correct bearing when facing north.
        """
        # Agent at (5, 5) facing north (θ = π/2)
        # Landmark directly to the right (east) at (7, 5)
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), pi / 2),
            obstacles=[],
            landmarks=[Landmark(Position(7, 5), 0)],
            timestep=0.1,
        )

        measurements = env.get_gt_to_landmarks()
        landmark = env.LANDMARKS[0]

        # Landmark is to the right, so bearing should be -π/2 (90° right)
        assert measurements[landmark].bearing == pytest.approx(-pi / 2, abs=1e-5)

    def test_get_gt_to_landmarks_correct_bearing_facing_east(self):
        """
        Test get_gt_to_landmarks computes correct bearing when facing east.
        """
        # Agent at (5, 5) facing east (θ = 0)
        # Landmark ahead at (7, 5)
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(7, 5), 0)],
            timestep=0.1,
        )

        measurements = env.get_gt_to_landmarks()
        landmark = env.LANDMARKS[0]

        # Landmark is directly ahead, bearing should be 0
        assert measurements[landmark].bearing == pytest.approx(0, abs=1e-5)

    def test_get_gt_to_landmarks_bearing_behind(self):
        """
        Test get_gt_to_landmarks computes correct bearing for landmark behind.
        """
        # Agent at (5, 5) facing east (θ = 0)
        # Landmark behind at (3, 5)
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(3, 5), 0)],
            timestep=0.1,
        )

        measurements = env.get_gt_to_landmarks()
        landmark = env.LANDMARKS[0]

        # Landmark is directly behind, bearing should be ±π
        bearing_mag = abs(measurements[landmark].bearing)
        assert bearing_mag == pytest.approx(pi, abs=1e-5)

    def test_get_gt_to_landmarks_bearing_wrapped(self, env: Environment):
        """
        Test get_gt_to_landmarks wraps bearing to [-π, π].
        """
        measurements = env.get_gt_to_landmarks()

        for _, br in measurements.items():
            assert br.bearing >= -pi
            assert br.bearing <= pi

    def test_get_gt_to_landmarks_zero_range(self):
        """
        Test get_gt_to_landmarks handles robot on landmark.
        """
        # Agent exactly on landmark position
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(5, 5), 0)],
            timestep=0.1,
        )

        measurements = env.get_gt_to_landmarks()
        landmark = env.LANDMARKS[0]

        # Range should be zero
        assert measurements[landmark].range == pytest.approx(0, abs=1e-5)

    def test_get_gt_to_landmarks_empty(self):
        """
        Test get_gt_to_landmarks returns empty dict with no landmarks.
        """
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[],
            timestep=0.1,
        )

        measurements = env.get_gt_to_landmarks()

        assert len(measurements) == 0
        assert isinstance(measurements, dict)

In [11]:
class TestRobotInitialization:
    """
    Tests for Robot initialization.
    """

    @pytest.fixture
    def env(self) -> Environment:
        """
        Fixture providing basic environment.
        """
        return Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(7, 7), 0)],
            timestep=0.1,
        )

    def test_initialization(self, env: Environment):
        """
        Robot initializes with correct default values.
        """
        robot = Robot(env)

        assert isinstance(robot.sensors, dict)
        assert len(robot.sensors) > 0
        assert robot.env is not None

    def test_has_required_sensors(self, env: Environment):
        """
        Robot initializes with GPS and LandmarkPinger.
        """
        robot = Robot(env)
        sensor_names = [s.name for s in robot.sensors.values()]

        assert "GPS" in sensor_names
        assert "LandmarkPinger" in sensor_names


class TestAgentStepDifferential:
    """
    Tests for differential drive math.
    """

    @pytest.fixture
    def robot(self) -> Robot:
        """
        Fixture providing robot in basic environment.
        """
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(7, 7), 0)],
            timestep=0.1,
        )
        return Robot(env)

    def test_differential_straight_line_motion(self, robot: Robot):
        """
        Robot moves forward in straight line with zero angular velocity.
        """
        initial_x = robot.env.robot_pose.pos.x
        initial_y = robot.env.robot_pose.pos.y
        initial_theta = robot.env.robot_pose.theta

        lin_vel = 1.0
        ang_vel = 0.0

        robot.robot_step_differential(lin_vel, ang_vel)

        dx = robot.env.robot_pose.pos.x - initial_x
        dy = robot.env.robot_pose.pos.y - initial_y
        distance_moved = sqrt(dx**2 + dy**2)

        expected_distance = lin_vel * robot.env.DT
        assert distance_moved == pytest.approx(expected_distance, abs=0.1)
        assert robot.env.robot_pose.theta == pytest.approx(initial_theta, abs=0.01)

    def test_differential_pure_rotation(self, robot: Robot):
        """
        Robot rotates in place with zero linear velocity.
        """
        initial_x = robot.env.robot_pose.pos.x
        initial_y = robot.env.robot_pose.pos.y
        initial_theta = robot.env.robot_pose.theta

        lin_vel = 0.0
        ang_vel = 1.0

        robot.robot_step_differential(lin_vel, ang_vel)

        dx = robot.env.robot_pose.pos.x - initial_x
        dy = robot.env.robot_pose.pos.y - initial_y
        distance_moved = sqrt(dx**2 + dy**2)
        assert distance_moved < 0.05

        expected_theta = initial_theta + ang_vel * robot.env.DT
        expected_theta = (expected_theta + pi) % (2 * pi) - pi
        assert robot.env.robot_pose.theta == pytest.approx(expected_theta, abs=0.1)

    def test_differential_arc_motion(self, robot: Robot):
        """
        Robot follows arc with both linear and angular velocity.
        """
        initial_pose = Pose(Position(5, 5), 0)
        robot.env.robot_pose = initial_pose

        lin_vel = 1.0
        ang_vel = 0.5

        robot.robot_step_differential(lin_vel, ang_vel)

        assert robot.env.robot_pose.pos.x != initial_pose.pos.x
        assert robot.env.robot_pose.pos.y != initial_pose.pos.y
        assert robot.env.robot_pose.theta != initial_pose.theta

    def test_differential_stores_velocities(self, robot: Robot):
        """
        Commanded velocities are stored for odometry.
        """
        lin_vel = 2.0
        ang_vel = 0.3

        robot.robot_step_differential(lin_vel, ang_vel)

        assert robot.cmd_lin_vel == pytest.approx(lin_vel, abs=0.2)
        assert robot.cmd_ang_vel == pytest.approx(ang_vel, abs=0.1)

    def test_differential_heading_wraps_correctly(self, robot: Robot):
        """
        Heading wraps to [-π, π] after large rotations.
        """
        robot.env.robot_pose = Pose(Position(5, 5), pi - 0.1)

        robot.robot_step_differential(0, 5.0)

        assert robot.env.robot_pose.theta >= -pi
        assert robot.env.robot_pose.theta <= pi

    def test_differential_respects_timestep(self):
        """
        Motion scales with environment timestep.
        """
        env_fast = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[],
            timestep=0.1,
        )
        env_slow = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[],
            timestep=0.01,
        )

        robot_fast = Robot(env_fast)
        robot_slow = Robot(env_slow)

        robot_fast.robot_step_differential(1.0, 0)
        robot_slow.robot_step_differential(1.0, 0)

        dist_fast = sqrt(
            (env_fast.robot_pose.pos.x - 5) ** 2 + (env_fast.robot_pose.pos.y - 5) ** 2
        )
        dist_slow = sqrt(
            (env_slow.robot_pose.pos.x - 5) ** 2 + (env_slow.robot_pose.pos.y - 5) ** 2
        )

        assert dist_fast > dist_slow

    def test_differential_backward_motion(self, robot: Robot):
        """
        Robot can move backward with negative linear velocity.
        """
        initial_x = robot.env.robot_pose.pos.x

        robot.robot_step_differential(-1.0, 0)

        assert robot.env.robot_pose.pos.x < initial_x

    def test_differential_negative_rotation(self, robot: Robot):
        """
        Robot can rotate clockwise with negative angular velocity.
        """
        initial_theta = robot.env.robot_pose.theta

        robot.robot_step_differential(0, -1.0)

        expected_theta = initial_theta - 1.0 * robot.env.DT
        expected_theta = (expected_theta + pi) % (2 * pi) - pi

        assert robot.env.robot_pose.theta == pytest.approx(expected_theta, abs=0.1)

    def test_differential_multiple_steps_accumulate(self, robot: Robot):
        """
        Multiple steps accumulate motion over time.
        """
        initial_x = robot.env.robot_pose.pos.x

        for _ in range(10):
            robot.robot_step_differential(1.0, 0)

        total_distance = robot.env.robot_pose.pos.x - initial_x
        assert total_distance > 0.5


# class TestAgentStepTranslational:
#     """Tests for Robot.agent_step_translational method"""

#     @pytest.fixture
#     def robot(self) -> Robot:
#         """Fixture providing robot in basic environment"""
#         env = Environment(
#             dimensions=Bounds(0, 10, 0, 10),
#             robot_pose=Pose(Position(5, 5), 0),
#             obstacles=[],
#             landmarks=[],
#             timestep=0.1,
#         )
#         return Robot(env)

#     def test_translational_x_motion(self, robot: Robot) -> None:
#         """Robot moves in x direction with translational drive."""
#         initial_x = robot.env.robot_pose.pos.x
#         initial_y = robot.env.robot_pose.pos.y

#         x_vel = 2.0
#         y_vel = 0.0

#         robot.agent_step_translational(x_vel, y_vel)

#         expected_dx = x_vel * robot.env.DT
#         actual_dx = robot.env.robot_pose.pos.x - initial_x
#         assert actual_dx == pytest.approx(expected_dx, abs=0.01)
#         assert robot.env.robot_pose.pos.y == pytest.approx(initial_y, abs=0.01)

#     def test_translational_y_motion(self, robot: Robot) -> None:
#         """Robot moves in y direction with translational drive."""
#         initial_x = robot.env.robot_pose.pos.x
#         initial_y = robot.env.robot_pose.pos.y

#         x_vel = 0.0
#         y_vel = 1.5

#         robot.agent_step_translational(x_vel, y_vel)

#         expected_dy = y_vel * robot.env.DT
#         actual_dy = robot.env.robot_pose.pos.y - initial_y
#         assert actual_dy == pytest.approx(expected_dy, abs=0.01)
#         assert robot.env.robot_pose.pos.x == pytest.approx(initial_x, abs=0.01)

#     def test_translational_diagonal_motion(self, robot: Robot) -> None:
#         """Robot moves diagonally with both x and y velocity."""
#         initial_pos = Position(5, 5)
#         robot.env.robot_pose = Pose(initial_pos, 0)

#         x_vel = 1.0
#         y_vel = 1.0

#         robot.agent_step_translational(x_vel, y_vel)

#         assert robot.env.robot_pose.pos.x != initial_pos.x
#         assert robot.env.robot_pose.pos.y != initial_pos.y

#         dx = robot.env.robot_pose.pos.x - initial_pos.x
#         dy = robot.env.robot_pose.pos.y - initial_pos.y
#         distance = sqrt(dx**2 + dy**2)
#         expected = sqrt((x_vel * robot.env.DT) ** 2 + (y_vel * robot.env.DT) ** 2)
#         assert distance == pytest.approx(expected, abs=0.01)

#     def test_translational_no_rotation(self, robot: Robot) -> None:
#         """Translational drive doesn't change heading."""
#         initial_theta = robot.env.robot_pose.theta

#         robot.agent_step_translational(1.0, 1.0)

#         assert robot.env.robot_pose.theta == initial_theta

#     def test_translational_negative_velocities(self, robot: Robot) -> None:
#         """Robot can move with negative velocities."""
#         initial_x = robot.env.robot_pose.pos.x
#         initial_y = robot.env.robot_pose.pos.y

#         robot.agent_step_translational(-1.0, -0.5)

#         assert robot.env.robot_pose.pos.x < initial_x
#         assert robot.env.robot_pose.pos.y < initial_y

#     def test_translational_zero_motion(self, robot: Robot) -> None:
#         """Zero velocity results in no motion."""
#         initial_pos = robot.env.robot_pose.pos

#         robot.agent_step_translational(0, 0)

#         assert robot.env.robot_pose.pos.x == initial_pos.x
#         assert robot.env.robot_pose.pos.y == initial_pos.y


class TestTakeSensorMeasurements:
    """
    Tests for taking sensor data.
    """

    @pytest.fixture
    def robot(self) -> Robot:
        """
        Fixture providing robot with sensors.
        """
        env = Environment(
            dimensions=Bounds(0, 10, 0, 10),
            robot_pose=Pose(Position(5, 5), 0),
            obstacles=[],
            landmarks=[Landmark(Position(7, 7), 0)],
            timestep=0.1,
        )
        return Robot(env)

    def test_sensor_measurements_has_all_sensors(self, robot: Robot):
        """
        Measurements dict contains all sensor names.
        """
        measurements = robot.take_sensor_measurements()

        for sensor in robot.sensors.values():
            assert sensor.name in measurements

    def test_sensor_measurements_respects_interval(self, robot: Robot):
        """
        Sensors only sample at their specified intervals.
        """
        robot.env.time = robot.sensors["GPS"].interval * 1

        measurements = robot.take_sensor_measurements()
        assert measurements["GPS"] is not None

        robot.env.time = robot.sensors["GPS"].interval * 1.5
        measurements = robot.take_sensor_measurements()
        assert measurements["GPS"] is None

        robot.env.time = robot.sensors["GPS"].interval * 2
        measurements = robot.take_sensor_measurements()
        assert measurements["GPS"] is not None

    def test_sensor_measurements_different_intervals(self, robot: Robot):
        """
        Sensors with different intervals sample independently.
        """
        robot.env.time = 0.0

        gps_samples = 0
        lp_samples = 0

        steps = 20

        for i in range(steps):
            robot.env.time = i * robot.env.DT  # two total seconds
            measurements = robot.take_sensor_measurements()

            if measurements["GPS"] is not None:
                gps_samples += 1
            if measurements["LandmarkPinger"] is not None:
                lp_samples += 1

        assert (
            lp_samples
            == steps * robot.env.DT / robot.sensors["LandmarkPinger"].interval
        )
        assert gps_samples == steps * robot.env.DT / robot.sensors["GPS"].interval

    def test_sensor_measurements_multiple_calls(self, robot: Robot):
        """
        Can call sensor measurements multiple times.
        """
        robot.env.time = 0.0

        m1 = robot.take_sensor_measurements()
        m2 = robot.take_sensor_measurements()

        assert isinstance(m1, dict)
        assert isinstance(m2, dict)
        assert len(m1) == len(m2)